# Sentiment Analysis of Airline Tweets
Imports and setup.

In [27]:
import os
import re
import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import TweetTokenizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

for p in ['stopwords','wordnet','vader_lexicon']:
    nltk.download(p, quiet=True)

os.makedirs('outputs', exist_ok=True)
sns.set_style('whitegrid')

Load the tweets and take a first look.

In [28]:
df = pd.read_csv('data/Tweets.csv')
print(df.shape)
df.head()

(14640, 15)


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [29]:
print(df['airline_sentiment'].value_counts())
print(df['airline'].value_counts())


print(df.isna().sum())

airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64
airline
United            3822
US Airways        2913
American          2759
Southwest         2420
Delta             2222
Virgin America     504
Name: count, dtype: int64
tweet_id                            0
airline_sentiment                   0
airline_sentiment_confidence        0
negativereason                   5462
negativereason_confidence        4118
airline                             0
airline_sentiment_gold          14600
name                                0
negativereason_gold             14608
retweet_count                       0
text                                0
tweet_coord                     13621
tweet_created                       0
tweet_location                   4733
user_timezone                    4820
dtype: int64


Clean the table. Keep useful columns, remove duplicates and fix the dates.

In [30]:
# keep only the columns we need
df = df[['tweet_id','airline','airline_sentiment','negativereason','text','tweet_created','user_timezone','retweet_count']].copy()

df = df.drop_duplicates(subset='tweet_id')
df=df.dropna(subset=['text','airline_sentiment'])
df = df.drop_duplicates(subset='text')     # same text posted twice

df['tweet_created'] = pd.to_datetime(df['tweet_created'], utc=True)
df['date']=df['tweet_created'].dt.date
df['user_timezone'] = df['user_timezone'].fillna('Unknown')

print(df.shape)
print(df['date'].min(), df['date'].max())

(14427, 9)
2015-02-17 2015-02-24


Preprocess the text. Tokenize, remove stop words and lemmatize. I keep words like "not" and "no" because they change the meaning.

In [31]:
tok = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
lemm = WordNetLemmatizer()

# keep negation words, they matter for sentiment
stop = set(stopwords.words('english')) - {'no','not','nor','never',"don't","didn't","doesn't","can't","won't"}


def clean_text(t):
    t = re.sub(r'@\w+', ' ', t)           # remove @mentions
    t = re.sub(r'http\S+', ' ', t)         #links
    t = t.replace('&amp;', 'and')
    t = re.sub(r'#', '', t)
    t = re.sub(r'[^a-zA-Z\' ]', ' ', t)
    words = tok.tokenize(t)
    words = [w for w in words if w not in stop and len(w)>1]
    words=[lemm.lemmatize(w) for w in words]
    return ' '.join(words)


df['clean_text'] = df['text'].apply(clean_text)
df = df[df['clean_text'].str.len()>0]

print(df[['text','clean_text']].head(5).to_string())

                                                                                                                             text                                                                  clean_text
0                                                                                             @VirginAmerica What @dhepburn said.                                                                        said
1                                                        @VirginAmerica plus you've added commercials to the experience... tacky.                                      plus added commercial experience tacky
2                                                         @VirginAmerica I didn't today... Must mean I need to take another trip!                               didn't today must mean need take another trip
3  @VirginAmerica it's really aggressive to blast obnoxious "entertainment" in your guests' faces &amp; they have little recourse  really aggressive blast obnoxious entertainme

Label the tweets with a pre-trained model (VADER) and see how close it is to the given labels.

In [32]:
vader = SentimentIntensityAnalyzer()

scores = df['text'].apply(vader.polarity_scores).apply(pd.Series)
scores.columns = ['v_neg','v_neu','v_pos','v_compound']
df = pd.concat([df, scores], axis=1)


def to_label(c):
    if c>=0.05: return 'positive'
    elif c<=-0.05: return 'negative'
    return 'neutral'


df['vader_label'] = df['v_compound'].apply(to_label)

print(df['vader_label'].value_counts())
print('agreement with given labels:', round(accuracy_score(df['airline_sentiment'], df['vader_label']),3))
print(pd.crosstab(df['airline_sentiment'], df['vader_label']))

vader_label
positive    5996
negative    5025
neutral     3386
Name: count, dtype: int64
agreement with given labels: 0.541
vader_label        negative  neutral  positive
airline_sentiment                             
negative               4512     1880      2683
neutral                 425     1297      1323
positive                 88      209      1990


Train a Logistic Regression model. It uses word and character TF-IDF plus the VADER scores. The given human labels are the ground truth.

In [33]:
vcols = ['v_neg','v_neu','v_pos','v_compound']
X = df[['clean_text']+vcols]
y = df['airline_sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)


def make_features():
    return ColumnTransformer([
        ('word', TfidfVectorizer(ngram_range=(1,2), min_df=2), 'clean_text'),
        ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=3, sublinear_tf=True), 'clean_text'),
        ('vader', 'passthrough', vcols)])


lr = make_pipeline(make_features(), LogisticRegression(max_iter=3000, C=3))
lr.fit(X_train, y_train)

pred = lr.predict(X_test)
acc = accuracy_score(y_test, pred)
print('test accuracy:', round(acc,3))
print(classification_report(y_test, pred))

(11525, 5) (2882, 5)
test accuracy: 0.803
              precision    recall  f1-score   support

    negative       0.85      0.91      0.88      1815
     neutral       0.65      0.57      0.61       609
    positive       0.77      0.68      0.72       458

    accuracy                           0.80      2882
   macro avg       0.76      0.72      0.74      2882
weighted avg       0.80      0.80      0.80      2882



Compare with other models, to see if ours is a good choice.

In [34]:
# other models to compare with
svm = make_pipeline(make_features(), LinearSVC(C=0.3))
svm.fit(X_train, y_train)
print('linear svm accuracy:', round(accuracy_score(y_test, svm.predict(X_test)),3))

nb = make_pipeline(ColumnTransformer([('w', TfidfVectorizer(ngram_range=(1,2), min_df=2), 'clean_text')]), MultinomialNB())
nb.fit(X_train, y_train)
print('naive bayes accuracy:', round(accuracy_score(y_test, nb.predict(X_test)),3))

# plain vader, no training at all
print('vader only accuracy:', round(accuracy_score(y_test, df.loc[X_test.index,'vader_label']),3))

linear svm accuracy: 0.808
naive bayes accuracy: 0.71
vader only accuracy: 0.54
